# Roles in Submission, Teaching, and Grading: Hacks

Original lesson: [Roles in Submission, Teaching, and Grading](https://pages.opencodingsociety.com/csa/unit_04/submission-roles) · My lesson plan: [Submission Roles]({{site.baseurl}}/sprint2/submission-roles)

## Submission Notes

<pre><code>Lesson: CSA Roles in Submission, Teaching, and Grading
MCQ Roles: 4/4 | answers: B,D,A,C
Popcorn: predicted counts (mine / teacher / all) = 2, 2, 3; actual = 2, 2, 3
Homework Hack 1: status() checks both print true (yes)
Homework Hack 2: countPending check prints true (yes)
Homework Hack 3: all three role checks print true (yes)
</code></pre>

All IDs on this page are made up. The page has no real student names, grades, or submission content.

## Popcorn Hack (In-Class)

I used my own IDs instead of the samples: `myId = 2027` for me, and `teacherId = 1201` for the teacher. I added two submissions where I am the submitter. One is for teacher 1201's assignment, and the other is for a different teacher, 1305.

### My prediction (written before running)

| Tab | Prediction | Why |
|-|-|-|
| My Submissions | **2** | Only my two rows have `submitterId == 2027`. Student 101's row does not count. |
| Assignments You Created (teacher 1201) | **2** | Student 101's row and my first row both have `assignmentOwnerId == 1201`. My row for teacher 1305 does not count. |
| All Submissions | **3** | The admin view does no filtering, so it shows every row in the list. |

In [1]:
import java.util.ArrayList;
import java.util.List;

public class PopcornRoles {
    static class Submission {
        int submitterId;
        int assignmentOwnerId;

        Submission(int submitterId, int assignmentOwnerId) {
            this.submitterId = submitterId;
            this.assignmentOwnerId = assignmentOwnerId;
        }
    }

    public static void main(String[] args) {
        int myId = 2027;       // my student ID
        int teacherId = 1201;  // the teacher whose assignment I submitted to

        List<Submission> submissions = new ArrayList<>();
        submissions.add(new Submission(101, teacherId));  // someone else's work

        // My two submissions: one for teacherId's assignment, one for a different teacher
        submissions.add(new Submission(myId, teacherId));
        submissions.add(new Submission(myId, 1305));

        int mine = 0;
        int teachers = 0;
        for (Submission submission : submissions) {
            if (submission.submitterId == myId) {
                mine++;
            }
            if (submission.assignmentOwnerId == teacherId) {
                teachers++;
            }
        }

        System.out.println("My Submissions tab: " + mine);
        System.out.println("Assignments You Created tab (teacher " + teacherId + "): " + teachers);
        System.out.println("All Submissions tab: " + submissions.size());
    }
}

PopcornRoles.main(null);

My Submissions tab: 2
Assignments You Created tab (teacher 1201): 2
All Submissions tab: 3

All three predictions matched: **2 / 2 / 3**. The teacher tab shows 2 because it compares only `assignmentOwnerId`, so it doesn't matter who submitted. Student 101's work and mine both count.

## MCQ Check

**Score: 4/4** · answers: B, D, A, C

| # | Answer | Why |
|-|-|-|
| 1 | **B.** The submission, with a Pending Review badge | The teacher branch checks only `assignmentOwnerId`, so the row is visible. A `null` `Double` grade means the submission hasn't been graded, not that its grade is 0.0, and it doesn't cause an error. |
| 2 | **D.** 2 | Submissions #1 and #3 in Code Example B both have `submitterId = 101`. |
| 3 | **A.** Admins see everything, and that shouldn't depend on any ID matching | An admin's `userId` usually doesn't match `submitterId` or `assignmentOwnerId`, so the admin rule has to be checked by itself, before any ID comparison. |
| 4 | **C.** Anyone can call the API directly | Filtering in the browser only controls what is *shown*. The server has to check the caller's role and identity, or a student could request the admin endpoint directly. |

## Homework Hack

### Hack 1: Add a Status Method to the Class

`grade` is a `Double`, which is a reference type, so `null` means "not graded yet." `status()` uses that null check to pick the string. The original starter printed `false` twice because it returned `"not implemented"`.

In [2]:
public class SubmissionStatusHack {
    static class Submission {
        Double grade;

        Submission(Double grade) {
            this.grade = grade;
        }

        // A null grade means no one has graded this submission yet
        String status() {
            if (grade != null) {
                return "graded";
            }
            return "pending";
        }
    }

    public static void main(String[] args) {
        Submission graded = new Submission(0.85);
        Submission pending = new Submission(null);

        System.out.println("graded".equals(graded.status()));
        System.out.println("pending".equals(pending.status()));
    }
}
SubmissionStatusHack.main(null);

true
true

### Hack 2: Count Pending Submissions

An enhanced `for` loop visits each submission and counts the ones whose `grade == null`. The loop only reads the list and never changes it. The starter returned `-1`, so its check printed `false`.

In [3]:
import java.util.ArrayList;
import java.util.List;

public class PendingCountHack {
    static class Submission {
        Double grade;

        Submission(Double grade) {
            this.grade = grade;
        }
    }

    public static int countPending(List<Submission> submissions) {
        int pending = 0;
        for (Submission submission : submissions) {
            if (submission.grade == null) {
                pending++;
            }
        }
        return pending;
    }

    public static void main(String[] args) {
        List<Submission> submissions = new ArrayList<>();
        submissions.add(new Submission(null));
        submissions.add(new Submission(0.75));
        submissions.add(new Submission(null));

        System.out.println(countPending(submissions) == 2);
    }
}
PendingCountHack.main(null);

true

### Hack 3: Filter Submissions by Role

This hack combines traversal and selection. `ADMIN` is checked first because an admin sees every row whatever the IDs are. Next, a teacher's rows are matched on `assignmentOwnerId` and a student's on `submitterId`. Matching rows are added to a **new** `visible` list. Removing rows from `allSubmissions` during the loop would throw a `ConcurrentModificationException`. Building a new list also leaves the shared original list unchanged for the other views.

In [4]:
import java.util.ArrayList;
import java.util.List;

public class RoleFilterHack {
    enum Role { STUDENT, TEACHER, ADMIN }

    static class Submission {
        int submitterId;
        int assignmentOwnerId;

        Submission(int submitterId, int assignmentOwnerId) {
            this.submitterId = submitterId;
            this.assignmentOwnerId = assignmentOwnerId;
        }
    }

    public static List<Submission> visibleSubmissionsFor(
            List<Submission> allSubmissions, Role role, int userId) {
        List<Submission> visible = new ArrayList<>();

        for (Submission submission : allSubmissions) {
            if (role == Role.ADMIN) {
                visible.add(submission);                    // All Submissions tab
            } else if (role == Role.TEACHER && submission.assignmentOwnerId == userId) {
                visible.add(submission);                    // Assignments You Created tab
            } else if (role == Role.STUDENT && submission.submitterId == userId) {
                visible.add(submission);                    // My Submissions tab
            }
        }

        return visible;
    }

    public static void main(String[] args) {
        List<Submission> allSubmissions = new ArrayList<>();
        allSubmissions.add(new Submission(101, 201));
        allSubmissions.add(new Submission(102, 201));
        allSubmissions.add(new Submission(103, 202));

        System.out.println(visibleSubmissionsFor(allSubmissions, Role.STUDENT, 101).size() == 1);
        System.out.println(visibleSubmissionsFor(allSubmissions, Role.TEACHER, 201).size() == 2);
        System.out.println(visibleSubmissionsFor(allSubmissions, Role.ADMIN, 0).size() == 3);
        System.out.println("Original list still has " + allSubmissions.size() + " submissions");
    }
}
RoleFilterHack.main(null);

true
true
true
Original list still has 3 submissions

**Security note:** `visibleSubmissionsFor` only decides what each role is *shown*. The real backend has to repeat the same role and identity check on the server before it returns any submissions.

## Self-Grade (Grading Plan, 1 point total)

| Part | Points | Self-grade | Evidence |
|-|-|-|-|
| Popcorn | 0.2 | 0.2 | Two submissions with my own ID (one for `teacherId`, one for another teacher). Predictions were written before running and match the output: 2 / 2 / 3. |
| MCQ | 0.2 | 0.2 | 4/4 (B, D, A, C) |
| Homework Hack 1 | 0.15 | 0.15 | `status()` returns `"graded"` or `"pending"` based on `grade`. Both checks print `true`. |
| Homework Hack 2 | 0.2 | 0.2 | `countPending` loops through the list and returns 2. The check prints `true`. |
| Homework Hack 3 | 0.25 | 0.25 | The loop with role-based selection filters all three roles correctly. All three checks print `true`. |
| **Total** | **1.0** | **1.0** | |

**Quick validation checklist:** each cell ends with `ClassName.main(null);` and shows output ✔ · MCQ score in the notes ✔ · Popcorn predictions written before running ✔ · every homework check prints `true` ✔ · no real student names, grades, or submission content ✔